# PySpark Zero to Hero — Master Reference Notebook

A single notebook covering all 27 lessons in the series. Run cells top-to-bottom — each section builds on the shared `SparkSession` and sample DataFrames created in **Section 1**.

> **Note:** Sections 9–11 (file I/O) require the files in `../datasets/`. Sections that need a cluster are shown with config patterns but run locally with `local[*]`. Section 23 (Delta Lake) stops and restarts the session.

| # | Section | Key APIs |
|---|---------|----------|
| 1 | Setup & SparkSession | `SparkSession.builder` |
| 2 | Creating DataFrames | `createDataFrame`, `printSchema` |
| 3 | Basic Transformations I | `select`, `col`, `expr`, `cast`, `alias` |
| 4 | Basic Transformations II | `withColumn`, `withColumnRenamed`, `drop` |
| 5 | Strings, Dates & Null Handling | `when`, `regexp_replace`, `to_date`, `coalesce` |
| 6 | Sort, Union & Aggregation | `orderBy`, `unionByName`, `groupBy`, `agg` |
| 7 | Distinct Values & Window Functions | `distinct`, `row_number`, `rank` |
| 8 | Joins & Partitioning | `join`, `repartition`, `coalesce` |
| 9 | Reading CSV Files | `read.format("csv")`, modes |
| 10 | Reading Parquet, ORC & Recursive | `read.format("parquet")`, `recursiveFileLookup` |
| 11 | Reading JSON | `from_json`, `explode`, `to_json` |
| 12 | Writing Data | `write.format().mode().save()`, `partitionBy` |
| 13 | User Defined Functions | `udf`, `spark.udf.register` |
| 14 | Execution Plans & DAG | `explain()` |
| 15 | Shuffle Optimization | `spark.sql.shuffle.partitions` |
| 16 | Caching & Persistence | `cache`, `persist`, `StorageLevel` |
| 17 | Broadcast Variables & Accumulators | `sparkContext.broadcast`, `accumulator` |
| 18 | Join Optimization | broadcast hints, bucketing |
| 19 | Adaptive Query Execution (AQE) | `spark.sql.adaptive.*` |
| 20 | Spark SQL & Temporary Views | `createOrReplaceTempView`, `spark.sql` |
| 21 | Concurrent Jobs | `ThreadPoolExecutor` |
| 22 | Memory Management | `spark.memory.fraction` |
| 23 | Delta Lake | ACID transactions, time travel, vacuum |

---
## Section 1 — Setup & SparkSession

In [ ]:
import concurrent.futures
import time
import random

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, lit, when,
    regexp_replace, to_date, date_format, current_date, current_timestamp,
    coalesce, asc, desc,
    count, sum, avg, max, min,
    row_number, rank, dense_rank,
    spark_partition_id, udf, broadcast,
    from_json, to_json, explode, concat, struct,
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType,
    ArrayType, LongType, BooleanType,
)
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

print('Imports OK')

In [ ]:
spark = (
    SparkSession.builder
    .appName("PySpark Zero to Hero — Master Notebook")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")  # keep low for local demos
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version        : {spark.version}")
print(f"Default parallelism  : {spark.sparkContext.defaultParallelism}")
print(f"Shuffle partitions   : {spark.conf.get('spark.sql.shuffle.partitions')}")

---
## Section 2 — Creating DataFrames

DataFrames can be built from Python lists (useful for tests/demos) or read from files. A **schema** makes the types explicit and avoids inference overhead.

In [ ]:
# ── Employee data ─────────────────────────────────────────────────────────────
emp_data = [
    ("E001", "D01", "John Doe",     30, "Male",   50000, "2015-01-01"),
    ("E002", "D01", "Jane Smith",   25, "Female", 45000, "2016-02-15"),
    ("E003", "D02", "Bob Johnson",  35, "Male",   60000, "2014-03-20"),
    ("E004", "D02", "Alice Brown",  28, "Female", 55000, "2017-04-10"),
    ("E005", "D03", "Charlie Lee",  40, "Male",   70000, "2013-05-05"),
    ("E006", "D03", "Diana Prince", 32, "Female", 65000, "2015-06-15"),
    ("E007", "D01", "Eve Wilson",   27, "Female", 48000, "2018-07-20"),
    ("E008", "D02", "Frank Miller", 45, "Male",   80000, "2012-08-01"),
    ("E009", "D03", "Grace Kelly",  29, "Female", 58000, "2019-09-10"),
    ("E010", "D04", "Henry Ford",   50, "Male",   90000, "2011-10-15"),
]

emp_schema = StructType([
    StructField("employee_id",   StringType(),  True),
    StructField("department_id", StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("age",           IntegerType(), True),
    StructField("gender",        StringType(),  True),
    StructField("salary",        IntegerType(), True),
    StructField("hire_date",     StringType(),  True),
])

emp_df = spark.createDataFrame(emp_data, emp_schema)

# ── Department data ───────────────────────────────────────────────────────────
dept_data = [
    ("D01", "Engineering"),
    ("D02", "Marketing"),
    ("D03", "Sales"),
    ("D04", "Finance"),
    ("D05", "HR"),          # no employees — useful for outer join demos
]
dept_schema = StructType([
    StructField("dept_id",   StringType(), True),
    StructField("dept_name", StringType(), True),
])
dept_df = spark.createDataFrame(dept_data, dept_schema)

# ── Sales data ────────────────────────────────────────────────────────────────
sales_data = [
    ("S001", "D01", "2024-01-15", 1500.0, "US"),
    ("S002", "D02", "2024-01-20", 2200.0, "UK"),
    ("S003", "D01", "2024-02-10", 1800.0, "US"),
    ("S004", "D03", "2024-02-25", 3000.0, "CA"),
    ("S005", "D02", "2024-03-05",  900.0, "US"),
]
sales_schema = StructType([
    StructField("sale_id",   StringType(), True),
    StructField("dept_id",   StringType(), True),
    StructField("sale_date", StringType(), True),
    StructField("amount",    DoubleType(), True),
    StructField("country",   StringType(), True),
])
sales_df = spark.createDataFrame(sales_data, sales_schema)

# ── Inspect ───────────────────────────────────────────────────────────────────
print("=== Employee DataFrame ===")
emp_df.printSchema()
emp_df.show()

print(f"Partitions: {emp_df.rdd.getNumPartitions()}")
print(f"Row count : {emp_df.count()}")

---
## Section 3 — Basic Transformations I: Select & Cast

`col()`, `expr()`, and `selectExpr()` are the three ways to reference and transform columns.

In [ ]:
# String column name — simple but no IDE support
emp_df.select("name", "salary").show(3)

# col() — preferred; enables chaining operators
emp_df.select(col("name"), col("salary") * 1.1).show(3)

# alias() — rename output column
emp_df.select(
    col("name"),
    (col("salary") * 1.1).alias("salary_plus_10pct"),
).show(3)

# expr() — SQL expression string; col() and expr() can be mixed freely
emp_df.select(
    expr("name"),
    expr("salary * 1.1 AS salary_raised"),
    expr("upper(name) AS name_upper"),
).show(3)

# selectExpr() — all columns as SQL strings (concise)
emp_df.selectExpr(
    "name",
    "salary * 1.1 AS salary_raised",
    "CAST(age AS STRING) AS age_str",
).show(3)

# Explicit cast via col()
emp_df.select(
    col("age").cast("string").alias("age_str"),
    col("salary").cast("double").alias("salary_dbl"),
).printSchema()

---
## Section 4 — Basic Transformations II: Add, Rename & Drop Columns

In [ ]:
# withColumn — add or overwrite a single column
emp_df2 = emp_df.withColumn("annual_bonus", col("salary") * 0.1)
emp_df2.select("name", "salary", "annual_bonus").show(3)

# lit() — constant / literal value
emp_df2 = emp_df2.withColumn("company", lit("Acme Corp"))
emp_df2.select("name", "company").show(3)

# withColumnRenamed
emp_df2 = emp_df2.withColumnRenamed("hire_date", "joining_date")
print(emp_df2.columns)

# withColumns — add multiple columns at once (Spark 3.3+)
emp_df3 = emp_df.withColumns({
    "salary_usd": col("salary").cast("double"),
    "is_senior":  when(col("age") >= 40, True).otherwise(False),
    "dept_label": concat(lit("Dept-"), col("department_id")),
})
emp_df3.select("name", "age", "salary_usd", "is_senior", "dept_label").show()

# drop — remove one or more columns
emp_df.drop("hire_date", "gender").show(3)

---
## Section 5 — Strings, Dates & Null Handling

In [ ]:
# ── Conditionals: when / otherwise (CASE WHEN) ────────────────────────────────
emp_df.withColumn(
    "salary_band",
    when(col("salary") >= 70000, "High")
    .when(col("salary") >= 55000, "Mid")
    .otherwise("Low"),
).select("name", "salary", "salary_band").show()

# ── String functions ───────────────────────────────────────────────────────────
emp_df.withColumn("name_underscored", regexp_replace(col("name"), " ", "_")) \
      .select("name", "name_underscored").show(3)

# ── Date functions ─────────────────────────────────────────────────────────────
(
    emp_df
    .withColumn("hire_dt",  to_date(col("hire_date"), "yyyy-MM-dd"))
    .withColumn("hire_year", expr("year(hire_dt)"))
    .withColumn("formatted", date_format(col("hire_dt"), "dd/MM/yyyy"))
    .select("name", "hire_date", "hire_dt", "hire_year", "formatted")
    .show()
)

spark.sql("SELECT current_date() AS today, current_timestamp() AS now").show(truncate=False)

In [ ]:
# Sample data with nulls
null_data = [
    ("E011", "D01", None,    None, "Male",   None,  "2020-01-01"),
    ("E012", "D02", "Tom C", 29,   None,    40000,  None),
]
null_df = spark.createDataFrame(null_data, emp_schema)
null_df.show()

# Drop rows where any column is null
null_df.na.drop().show()

# Drop rows only where name is null
null_df.na.drop(subset=["name"]).show()

# Fill nulls with defaults per column
null_df.na.fill({"name": "Unknown", "salary": 0, "gender": "N/A"}).show()

# coalesce — return first non-null across columns
null_df.withColumn("safe_salary", coalesce(col("salary"), lit(0))) \
       .select("name", "salary", "safe_salary").show()

---
## Section 6 — Sorting, Union & Aggregation

In [ ]:
# ── Sorting ────────────────────────────────────────────────────────────────────
emp_df.orderBy(col("salary").desc()).select("name", "salary").show()

# Multi-column sort
emp_df.orderBy(asc("department_id"), desc("salary")) \
      .select("department_id", "name", "salary").show()

# ── Union ──────────────────────────────────────────────────────────────────────
extra_emp = spark.createDataFrame([
    ("E011", "D04", "Ivan Drago",  38, "Male",   72000, "2020-11-01"),
    ("E012", "D01", "Julia Child", 44, "Female", 85000, "2021-03-15"),
], emp_schema)

# unionByName — matches columns by name (safe if schemas differ in order)
all_emp = emp_df.unionByName(extra_emp)
print(f"Row count after union: {all_emp.count()}")

# ── Aggregation ────────────────────────────────────────────────────────────────
emp_df.groupBy("department_id").agg(
    count("employee_id").alias("headcount"),
    avg("salary").alias("avg_salary"),
    sum("salary").alias("total_salary"),
    max("age").alias("oldest"),
).orderBy("department_id").show()

# HAVING equivalent — filter the aggregated result
emp_df.groupBy("department_id") \
      .agg(count("*").alias("cnt")) \
      .where(col("cnt") >= 3) \
      .show()

---
## Section 7 — Distinct Values & Window Functions

Window functions compute per-row results within a group (partition) without collapsing rows — unlike `groupBy`.

In [ ]:
# ── Distinct ───────────────────────────────────────────────────────────────────
emp_df.select("department_id").distinct().show()

# dropDuplicates on a subset
emp_df.dropDuplicates(["department_id"]).select("department_id", "name").show()

# ── Window functions ───────────────────────────────────────────────────────────
window_dept = Window.partitionBy("department_id").orderBy(desc("salary"))

(
    emp_df
    .withColumn("row_num",    row_number().over(window_dept))  # unique, no ties
    .withColumn("rank",       rank().over(window_dept))        # gaps on ties
    .withColumn("dense_rank", dense_rank().over(window_dept))  # no gaps
    .select("department_id", "name", "salary", "row_num", "rank", "dense_rank")
    .orderBy("department_id", "row_num")
    .show()
)

# Running max salary within department
emp_df.withColumn(
    "dept_max_salary",
    max("salary").over(Window.partitionBy("department_id")),
).select("department_id", "name", "salary", "dept_max_salary").show()

# Top earner per department
emp_df.withColumn("rn", row_number().over(window_dept)) \
      .filter(col("rn") == 1) \
      .select("department_id", "name", "salary") \
      .show()

---
## Section 8 — Joins & Data Partitioning

`repartition()` shuffles data (even distribution by key). `coalesce()` merges partitions without a full shuffle (faster, but can create skew).

In [ ]:
# repartition — full shuffle, target number of partitions
rep4 = emp_df.repartition(4)
print(f"repartition(4) : {rep4.rdd.getNumPartitions()} partitions")

# coalesce — merge partitions, no shuffle
coal2 = rep4.coalesce(2)
print(f"coalesce(2)    : {coal2.rdd.getNumPartitions()} partitions")

# repartition by column — same department_id lands in the same partition
by_dept = emp_df.repartition(4, "department_id")
by_dept.groupBy(spark_partition_id().alias("partition"), "department_id") \
       .count().orderBy("partition").show()

In [ ]:
# ── Inner join ─────────────────────────────────────────────────────────────────
emp_df.join(dept_df, emp_df["department_id"] == dept_df["dept_id"], how="inner") \
      .select("name", "department_id", "dept_name", "salary").show()

# ── Left outer join — retains all employees, D05 has no employees ──────────────
emp_df.alias("e").join(
    dept_df.alias("d"),
    col("e.department_id") == col("d.dept_id"),
    how="left",
).select("e.name", "d.dept_name", "e.salary").show()

# D05/HR appears with nulls
dept_df.alias("d").join(
    emp_df.alias("e"),
    col("d.dept_id") == col("e.department_id"),
    how="left",
).select("d.dept_name", "e.name", "e.salary").orderBy("dept_name").show()

# ── Complex join condition (AND) ───────────────────────────────────────────────
emp_df.alias("e").join(
    dept_df.alias("d"),
    (col("e.department_id") == col("d.dept_id")) & (col("e.salary") > 55000),
    how="inner",
).select("e.name", "d.dept_name", "e.salary").show()

---
## Section 9 — Reading CSV Files

Common options: `header`, `inferSchema` (slow on large files — prefer explicit schema), `mode` for bad-record handling.

In [ ]:
CSV_PATH = "../datasets/emp.csv"

# ── Basic read ─────────────────────────────────────────────────────────────────
df_csv = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(CSV_PATH)
)
df_csv.printSchema()
df_csv.show(5)

# ── Explicit schema (preferred in production) ──────────────────────────────────
explicit_schema = StructType([
    StructField("employee_id",   StringType(),  True),
    StructField("department_id", StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("age",           IntegerType(), True),
    StructField("gender",        StringType(),  True),
    StructField("salary",        IntegerType(), True),
    StructField("hire_date",     StringType(),  True),
])

df_csv_typed = (
    spark.read.format("csv")
    .option("header", True)
    .schema(explicit_schema)
    .load(CSV_PATH)
)
df_csv_typed.printSchema()

# ── Error handling modes ───────────────────────────────────────────────────────
# PERMISSIVE  (default) — bad rows → null + captured in _corrupt column
# DROPMALFORMED         — skip bad rows silently
# FAILFAST              — throw exception on first bad row

corrupt_schema = explicit_schema.add("_corrupt", StringType(), True)
df_permissive = (
    spark.read.format("csv")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt")
    .schema(corrupt_schema)
    .load(CSV_PATH)
)
# Show only the bad rows (none expected in clean emp.csv)
df_permissive.filter(col("_corrupt").isNotNull()).show()

print(f"Total rows read: {df_csv_typed.count()}")

---
## Section 10 — Reading Parquet, ORC & Recursive Files

Columnar formats embed schema, support predicate pushdown, and compress far better than CSV. Parquet is the default Spark format.

In [ ]:
PARQUET_PATH   = "../datasets/sales_data.parquet"
ORC_PATH       = "../datasets/sales_data.orc"
RECURSIVE_PATH = "../datasets/sales_recursive"

# ── Parquet ────────────────────────────────────────────────────────────────────
df_parquet = spark.read.format("parquet").load(PARQUET_PATH)
df_parquet.printSchema()
df_parquet.show(5)

# Column pruning — Spark only reads columns needed (physical skip)
df_parquet.select("sale_id", "amount").show(5)

# ── ORC ────────────────────────────────────────────────────────────────────────
df_orc = spark.read.format("orc").load(ORC_PATH)
df_orc.printSchema()
df_orc.show(5)

# ── Recursive file lookup — reads all nested subdirectories ───────────────────
df_recursive = (
    spark.read.format("parquet")
    .option("recursiveFileLookup", True)
    .load(RECURSIVE_PATH)
)
print(f"Rows from recursive read: {df_recursive.count()}")
df_recursive.show(5)

---
## Section 11 — Reading JSON Files

JSON can be single-line (one JSON object per line) or multi-line. `from_json` parses a JSON string column; `explode` flattens arrays into rows.

In [ ]:
JSON_SL = "../datasets/order_singleline.json"   # one JSON object per line
JSON_ML = "../datasets/order_multiline.json"     # single JSON array across lines

# ── Single-line JSON ───────────────────────────────────────────────────────────
df_json_sl = spark.read.format("json").load(JSON_SL)
df_json_sl.printSchema()
df_json_sl.show(truncate=False)

# ── Multi-line JSON ────────────────────────────────────────────────────────────
df_json_ml = (
    spark.read.format("json")
    .option("multiLine", True)
    .load(JSON_ML)
)
df_json_ml.printSchema()
df_json_ml.show(truncate=False)

# ── Parse a JSON string column with from_json ──────────────────────────────────
raw_data = [
    ("ORD-1", '{"items":["Laptop","Mouse"],"total":1200}'),
    ("ORD-2", '{"items":["Desk"],"total":450}'),
]
raw_df = spark.createDataFrame(raw_data, ["order_id", "payload"])

payload_schema = StructType([
    StructField("items", ArrayType(StringType()), True),
    StructField("total", IntegerType(),           True),
])
parsed = raw_df.withColumn("data", from_json(col("payload"), payload_schema))
parsed.select("order_id", "data.*").show(truncate=False)

# explode — one row per array element
parsed.withColumn("item", explode(col("data.items"))) \
      .select("order_id", "item", col("data.total").alias("total")).show()

# to_json — convert a struct back to a JSON string
parsed.withColumn("json_out", to_json(col("data"))) \
      .select("order_id", "json_out").show(truncate=False)

---
## Section 12 — Writing Data

| Mode | Behaviour |
|------|-----------|
| `error` | Fail if path exists (default) |
| `overwrite` | Replace existing data |
| `append` | Add to existing data |
| `ignore` | Skip silently if path exists |

In [ ]:
OUTPUT = "/tmp/spark_master_nb"

# ── CSV ────────────────────────────────────────────────────────────────────────
emp_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", True) \
    .save(f"{OUTPUT}/emp_csv")

# ── Parquet (default compression: snappy) ─────────────────────────────────────
emp_df.write.format("parquet") \
    .mode("overwrite") \
    .save(f"{OUTPUT}/emp_parquet")

# ── Partitioned write — creates dept_id=D01/, dept_id=D02/ … subdirs ──────────
emp_df.write.format("parquet") \
    .mode("overwrite") \
    .partitionBy("department_id") \
    .save(f"{OUTPUT}/emp_by_dept")

# ── Single output file (repartition(1) before write) ──────────────────────────
emp_df.repartition(1).write.format("csv") \
    .mode("overwrite") \
    .option("header", True) \
    .save(f"{OUTPUT}/emp_single")

# ── Read back partitioned data ─────────────────────────────────────────────────
df_back = (
    spark.read.format("parquet")
    .option("recursiveFileLookup", True)
    .load(f"{OUTPUT}/emp_by_dept")
)
print(f"Rows read back from partitioned write: {df_back.count()}")
df_back.show()

---
## Section 13 — User Defined Functions (UDFs)

UDFs let you apply arbitrary Python logic column-by-column. They are slower than native Spark functions (row-by-row serialization), so prefer built-in functions where possible.

In [ ]:
# ── Define and register a UDF ──────────────────────────────────────────────────
def salary_grade(salary):
    if salary is None:
        return "Unknown"
    if salary >= 70000:
        return "Grade A"
    elif salary >= 55000:
        return "Grade B"
    else:
        return "Grade C"

salary_grade_udf = udf(salary_grade, StringType())

emp_df.withColumn("grade", salary_grade_udf(col("salary"))) \
      .select("name", "salary", "grade").show()

# ── Decorator style ────────────────────────────────────────────────────────────
@udf(returnType=StringType())
def initials_udf(name):
    if name is None:
        return None
    return ".".join(part[0].upper() for part in name.split()) + "."

emp_df.withColumn("initials", initials_udf(col("name"))) \
      .select("name", "initials").show()

# ── Register for use inside spark.sql() ───────────────────────────────────────
spark.udf.register("salary_grade_sql", salary_grade, StringType())
emp_df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT name, salary, salary_grade_sql(salary) AS grade
    FROM   employees
    ORDER  BY salary DESC
""").show()

---
## Section 14 — Execution Plans & DAG

`explain()` prints the logical and physical plans. Look for `Exchange` (shuffle), `BroadcastHashJoin`, and `SortMergeJoin` nodes.

In [ ]:
# Disable AQE so plans are deterministic and readable
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force SortMerge

print("=== Simple filter plan ===")
emp_df.filter(col("salary") > 50000).explain()

print("=== Join plan (SortMergeJoin with Exchange shuffles) ===")
emp_df.join(dept_df, emp_df["department_id"] == dept_df["dept_id"]).explain()

print("=== Full plan (logical + physical) ===")
emp_df.groupBy("department_id").agg(avg("salary")).explain(True)

# Restore defaults
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

---
## Section 15 — Shuffle Optimization

Shuffle is the most expensive operation in Spark. Two levers:
1. **`spark.sql.shuffle.partitions`** — controls how many partitions are created after a shuffle (default 200, far too many for small data).
2. **Pre-partitioning** on the join key — if both sides are already partitioned identically, Spark can skip the shuffle Exchange entirely.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

print("Default shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Too many partitions for small data → many tiny tasks
spark.conf.set("spark.sql.shuffle.partitions", "200")
emp_df.groupBy("department_id").agg(count("*")).explain()

# Tuned for the data size
spark.conf.set("spark.sql.shuffle.partitions", "4")
emp_df.groupBy("department_id").agg(count("*")).show()

# Pre-partition both sides on the join key → no Exchange in the plan
emp_part  = emp_df.repartition(4, "department_id")
dept_part = dept_df.repartition(4, "dept_id")

print("=== Join plan WITH pre-partitioned inputs ===")
emp_part.join(dept_part, emp_part["department_id"] == dept_part["dept_id"]).explain()

spark.conf.set("spark.sql.adaptive.enabled", "true")

---
## Section 16 — Caching & Persistence

Use caching when the same DataFrame is used in multiple downstream actions. `cache()` is shorthand for `MEMORY_AND_DISK`.

| Storage Level | Description |
|---------------|-------------|
| `MEMORY_ONLY` | Fastest; evicts if no space |
| `MEMORY_AND_DISK` | Spills to disk when memory full (safest default) |
| `MEMORY_ONLY_SER` | Serialized in memory (smaller footprint, slower access) |
| `MEMORY_AND_DISK_2` | Replicated on 2 nodes (fault-tolerant) |
| `DISK_ONLY` | Always on disk |

In [ ]:
# cache() — equivalent to MEMORY_AND_DISK
emp_df.cache()
emp_df.count()   # first action triggers caching
emp_df.count()   # subsequent actions hit cache

# persist() with explicit level
sales_df.persist(StorageLevel.MEMORY_AND_DISK)
sales_df.count()

# Check what is cached
print(spark.catalog.listTables())  # shows registered views only

# Unpersist a specific DataFrame
emp_df.unpersist()
sales_df.unpersist()

# Clear all cached tables and DataFrames
spark.catalog.clearCache()
print("Cache cleared")

---
## Section 17 — Broadcast Variables & Accumulators

**Broadcast variables** ship a read-only value to every executor once (vs. resending with every task). Ideal for lookup tables < a few hundred MB.

**Accumulators** aggregate values across partitions back to the driver. Only the driver can read the value.

In [ ]:
# ── Broadcast variable — dict lookup ──────────────────────────────────────────
dept_lookup = {"D01": "Engineering", "D02": "Marketing",
               "D03": "Sales",       "D04": "Finance"}
bc_dept = spark.sparkContext.broadcast(dept_lookup)

@udf(returnType=StringType())
def lookup_dept(dept_id):
    return bc_dept.value.get(dept_id, "Unknown")

emp_df.withColumn("dept_name", lookup_dept(col("department_id"))) \
      .select("name", "department_id", "dept_name").show()

# ── Accumulator — count seniors ───────────────────────────────────────────────
senior_count = spark.sparkContext.accumulator(0)
high_sal_sum = spark.sparkContext.accumulator(0)

def analyse_row(row):
    if row.age >= 40:
        senior_count.add(1)
    if row.salary >= 70000:
        high_sal_sum.add(row.salary)

emp_df.foreach(analyse_row)

print(f"Employees aged 40+        : {senior_count.value}")
print(f"Total salary (>= 70k)     : {high_sal_sum.value}")

---
## Section 18 — Join Optimization: Broadcast Hints & Bucketing

**Broadcast join** ships the smaller table to every executor — eliminates the shuffle Exchange entirely. Use when one side fits in memory.

**Bucketing** pre-sorts data on disk by the join key. Two bucketed tables joined on the same key and bucket count skip the shuffle at read time.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force SortMerge

print("=== SortMergeJoin (default with threshold disabled) ===")
emp_df.join(dept_df, emp_df["department_id"] == dept_df["dept_id"]).explain()

print("=== BroadcastHashJoin (explicit broadcast() hint) ===")
emp_df.join(broadcast(dept_df), emp_df["department_id"] == dept_df["dept_id"]).explain()

# SQL broadcast hint
dept_df.createOrReplaceTempView("dept")
emp_df.createOrReplaceTempView("emp")
spark.sql("""
    SELECT /*+ BROADCAST(dept) */ e.name, d.dept_name
    FROM   emp  e
    JOIN   dept d ON e.department_id = d.dept_id
""").explain()

# ── Bucketing (requires Hive Metastore — config pattern shown) ─────────────────
# emp_df.write  \
#     .bucketBy(4, "department_id") \
#     .sortBy("department_id") \
#     .saveAsTable("emp_bucketed")
#
# dept_df.write \
#     .bucketBy(4, "dept_id") \
#     .sortBy("dept_id") \
#     .saveAsTable("dept_bucketed")
#
# spark.table("emp_bucketed") \
#     .join(spark.table("dept_bucketed"),
#           "department_id == dept_id") \
#     .explain()   # ← no Exchange nodes

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

---
## Section 19 — Adaptive Query Execution (AQE)

AQE (enabled by default in Spark 3.2+) re-optimises plans at runtime using shuffle statistics. Three key features:
1. **Coalesce post-shuffle partitions** — merge small partitions after a shuffle.
2. **Skew join handling** — split oversized partitions automatically.
3. **Runtime broadcast conversion** — convert a SortMerge to BroadcastHash if one side turns out small.

In [ ]:
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

# ── Coalesce small post-shuffle partitions ─────────────────────────────────────
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", str(64 * 1024 * 1024))  # 64 MB target

# ── Skew join ─────────────────────────────────────────────────────────────────
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", str(256 * 1024 * 1024))  # 256 MB

# ── Runtime broadcast conversion ──────────────────────────────────────────────
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10 MB

# Demonstrate: AQE converts to BroadcastHashJoin at runtime
print("=== Plan with AQE enabled ===")
emp_df.join(dept_df, emp_df["department_id"] == dept_df["dept_id"]).explain()

# Compare: same join without AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")
print("=== Plan without AQE ===")
emp_df.join(dept_df, emp_df["department_id"] == dept_df["dept_id"]).explain()

spark.conf.set("spark.sql.adaptive.enabled", "true")

---
## Section 20 — Spark SQL & Temporary Views

Temp views let you query DataFrames with SQL. They are session-scoped and disappear when the session ends.

In [ ]:
# Register views
emp_df.createOrReplaceTempView("employees")
dept_df.createOrReplaceTempView("departments")
sales_df.createOrReplaceTempView("sales")

# ── Basic query ────────────────────────────────────────────────────────────────
spark.sql("""
    SELECT name, salary
    FROM   employees
    WHERE  salary > 55000
    ORDER  BY salary DESC
""").show()

# ── Join ───────────────────────────────────────────────────────────────────────
spark.sql("""
    SELECT e.name, d.dept_name, e.salary
    FROM   employees  e
    JOIN   departments d ON e.department_id = d.dept_id
    WHERE  e.salary > 50000
""").show()

# ── Aggregation with HAVING ────────────────────────────────────────────────────
spark.sql("""
    SELECT d.dept_name,
           COUNT(*)      AS headcount,
           AVG(e.salary) AS avg_salary
    FROM   employees  e
    JOIN   departments d ON e.department_id = d.dept_id
    GROUP  BY d.dept_name
    HAVING COUNT(*) >= 2
    ORDER  BY avg_salary DESC
""").show()

# ── Window function in SQL ─────────────────────────────────────────────────────
spark.sql("""
    SELECT name, department_id, salary,
           RANK()       OVER (PARTITION BY department_id ORDER BY salary DESC) AS rnk,
           ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rn
    FROM employees
""").show()

# ── Broadcast hint in SQL ──────────────────────────────────────────────────────
spark.sql("""
    SELECT /*+ BROADCAST(departments) */ e.name, d.dept_name
    FROM   employees  e
    JOIN   departments d ON e.department_id = d.dept_id
""").explain()

---
## Section 21 — Concurrent Jobs with ThreadPoolExecutor

Multiple Spark jobs can run concurrently from the same `SparkSession` using Python threads. This is useful when processing independent partitions or files in parallel.

In [ ]:
cities_data = [
    ("New York",    "US", 8336817),
    ("Los Angeles", "US", 3979576),
    ("Chicago",     "US", 2693976),
    ("London",      "UK", 9541000),
    ("Manchester",  "UK", 2756000),
    ("Toronto",     "CA", 2731571),
    ("Vancouver",   "CA",  631486),
]
cities_df = spark.createDataFrame(cities_data, ["city", "country", "population"])
cities_df.cache()
cities_df.count()  # trigger caching

countries = [r.country for r in cities_df.select("country").distinct().collect()]

def process_country(country):
    t0 = time.time()
    (
        cities_df
        .filter(col("country") == country)
        .agg(
            count("city").alias("city_count"),
            sum("population").alias("total_pop"),
        )
        .withColumn("country", lit(country))
        .collect()
    )
    return country, round(time.time() - t0, 3)

# Sequential
t0 = time.time()
for c in countries:
    process_country(c)
seq_time = round(time.time() - t0, 2)
print(f"Sequential : {seq_time}s")

# Concurrent
t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=len(countries)) as pool:
    futures = {pool.submit(process_country, c): c for c in countries}
    for f in concurrent.futures.as_completed(futures):
        country, elapsed = f.result()
        print(f"  {country}: {elapsed}s")
conc_time = round(time.time() - t0, 2)
print(f"Concurrent : {conc_time}s  (speedup ≈ {seq_time / max(conc_time, 0.01):.1f}x)")

cities_df.unpersist()

---
## Section 22 — Memory Management

Understanding how Spark divides executor memory helps prevent OOM errors.

```
┌─────────────────────────────────────────────────────────────────┐
│  JVM Heap  =  spark.executor.memory  (e.g. 512 MB)             │
│    Reserved memory        =  300 MB  (JVM overhead — fixed)     │
│    Usable memory          =  212 MB  (Heap − 300 MB)            │
│      Spark managed memory =  127 MB  (60% of usable)  ← fraction│
│        Storage pool       =   64 MB  (50% of Spark mem)         │
│        Execution pool     =   63 MB  (50% of Spark mem)         │
│      User memory          =   85 MB  (40% of usable)            │
└─────────────────────────────────────────────────────────────────┘
```

- **Storage pool** — cached DataFrames (`cache()` / `persist()`)
- **Execution pool** — shuffle, sort, join buffers
- Pools borrow from each other dynamically (unified memory model)
- OOM usually means execution memory pressure; try reducing shuffle partitions, enabling spill, or increasing executor memory

In [ ]:
# Inspect current memory config
for key in [
    "spark.executor.memory",
    "spark.memory.fraction",
    "spark.memory.storageFraction",
    "spark.driver.memory",
]:
    try:
        print(f"{key:45s} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key:45s} = (using default)")

# Tune memory fractions at session level
spark.conf.set("spark.memory.fraction",        "0.6")  # 60% of usable heap for Spark
spark.conf.set("spark.memory.storageFraction", "0.5")  # 50% of Spark mem reserved for storage
print("\nTuned memory fractions applied.")

# ── Typical OOM scenarios ──────────────────────────────────────────────────────
# 1. Caching too much — reduce StorageLevel or evict unneeded DataFrames
# 2. Shuffle with many large partitions — reduce spark.sql.shuffle.partitions
#    or increase spark.executor.memory
# 3. Collect() on large data — use .show() / write to disk instead
# 4. UDFs loading large objects — use broadcast variables instead

---
## Section 23 — Delta Lake: ACID Transactions & Time Travel

Delta Lake adds ACID semantics, versioned history, and time travel on top of Parquet files. It requires special SparkSession extensions.

> **Note:** The current session is stopped below and restarted with Delta Lake configured. All previous DataFrames will be gone after `spark.stop()`.

In [ ]:
# Stop the existing session
spark.stop()
print("Previous session stopped.")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .appName("Delta Lake Demo")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

from delta import DeltaTable

print(f"Delta Lake session ready — Spark {spark.version}")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

DELTA_PATH = "/tmp/spark_master_nb/delta_sales"

# ── Version 0 — initial write ─────────────────────────────────────────────────
sales_schema = StructType([
    StructField("sale_id", StringType(), True),
    StructField("rep",     StringType(), True),
    StructField("country", StringType(), True),
    StructField("amount",  DoubleType(), True),
])
sales_v0 = spark.createDataFrame([
    ("S001", "Alice", "US", 1500.0),
    ("S002", "Bob",   "UK", 2200.0),
    ("S003", "Carol", "CA", 3000.0),
    ("S004", "Dave",  "US",  900.0),
], sales_schema)

sales_v0.write.format("delta").mode("overwrite").save(DELTA_PATH)
print("Version 0 written")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# ── Version 1 — append ────────────────────────────────────────────────────────
spark.createDataFrame([
    ("S005", "Eve", "AU", 4100.0),
], sales_schema).write.format("delta").mode("append").save(DELTA_PATH)

print("Version 1 — after append")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
dt = DeltaTable.forPath(spark, DELTA_PATH)

# ── Version 2 — UPDATE ────────────────────────────────────────────────────────
dt.update(
    condition="rep = 'Alice'",
    set={"amount": "amount * 1.10"},   # 10% raise
)
print("Version 2 — after UPDATE Alice's amount")
spark.read.format("delta").load(DELTA_PATH).show()

# ── Version 3 — DELETE ───────────────────────────────────────────────────────
dt.delete(condition="amount < 1000")
print("Version 3 — after DELETE where amount < 1000")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# ── History ────────────────────────────────────────────────────────────────────
dt.history().select("version", "timestamp", "operation").show(truncate=False)

# ── Time travel — read older versions ─────────────────────────────────────────
print("=== Version 0 (original) ===")
spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH).show()

print("=== Version 1 (after append) ===")
spark.read.format("delta").option("versionAsOf", 1).load(DELTA_PATH).show()

# ── RESTORE ────────────────────────────────────────────────────────────────────
dt.restoreToVersion(1)
print("Restored to version 1")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# ── VACUUM — remove old data files outside the retention window ────────────────
# Default retention = 7 days (168 hours). Lowering below 7 days is unsafe in
# production unless you are certain no running queries need older files.

# In production:
# spark.sql(f"VACUUM delta.`{DELTA_PATH}` RETAIN 168 HOURS")

# For demo — override the safety check:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
dt.vacuum(retentionHours=0)   # purge all obsolete files
print("VACUUM complete")

# ── OPTIMIZE & Z-ORDER (Databricks / OSS Delta) ───────────────────────────────
# Compacts small files into larger ones for faster reads:
# spark.sql(f"OPTIMIZE delta.`{DELTA_PATH}`")
#
# Z-Order clusters data by the specified columns — improves selective queries:
# spark.sql(f"OPTIMIZE delta.`{DELTA_PATH}` ZORDER BY (country)")

print("\nDelta Lake demo complete!")
spark.stop()